In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\saiou\Desktop\mapping_visualisation_project\world-population-interactive-visualisation


In [4]:

import json
from mcp import Client
from agentic_framework.mcp_client import MCP_URL

async with Client(MCP_URL) as client:
    response = await client.list_tools()

    for tool in response.tools:
        if tool.name in {
            "data360_search_indicators",
            "data360_get_metadata",
            "data360_get_data",
        }:
            print(f"\n{'=' * 70}")
            print("TOOL:", tool.name)
            print("DESCRIPTION:", tool.description)
            print("\nINPUT SCHEMA:")
            print(json.dumps(tool.input_schema, indent=2))



TOOL: data360_search_indicators
DESCRIPTION: Search for Data360 indicators with enriched metadata for selection.

Use when the user asks for data on a development topic (e.g. GDP, poverty, education).
Default to using the single `query` parameter for any single topic/indicator search. Use the `queries` or `query_groups` parameters ONLY when the request involves multiple topics or scopes (2 or more).
Provide exactly one of `query`, `queries`, or `query_groups`. One of these is strictly required.

### Parameter Selection Decision Tree (CRITICAL):
1. **Exactly 1 Topic** (e.g., "life expectancy" or "mortality rate") for any number of countries → you MUST use the single `query` parameter + `required_country`. Do NOT use `queries` with only one element, as it will fail. Do NOT combine multiple topics with 'and' or 'or' in `query` (e.g. do NOT use query="GDP and inflation").
2. **Multiple Topics, Same Country/Countries** (e.g., "life expectancy and GDP per capita" for Japan) → you MUST use t

In [5]:

from mcp import Client
from agentic_framework.mcp_client import (
    MCP_URL,
    call_data360_tool,
)

search_cases = {
    "life_expectancy": {
        "query": "life expectancy at birth total",
        "country": "EGY",
    },
    "co2_emissions": {
        "query": "CO2 emissions per capita",
        "country": "FRA",
    },
    "unemployment": {
        "query": "unemployment total modeled ILO estimate",
        "country": "CHE",
    },
}

search_results = {}

async with Client(MCP_URL) as client:
    for case_name, case in search_cases.items():

        result = await call_data360_tool(
            client,
            "data360_search_indicators",
            {
                "query": case["query"],
                "required_country": case["country"],
                "database": "WB_WDI",
                "limit": 10,
                "offset": 0,
            },
        )

        search_results[case_name] = result

        print(f"\n{'=' * 70}")
        print("CASE:", case_name)
        print("SEARCH:", case["query"])
        print("COUNTRY:", case["country"])
        print("\nRESULT:")
        print(json.dumps(result, indent=2, default=str)[:15000])



CASE: life_expectancy
SEARCH: life expectancy at birth total
COUNTRY: EGY

RESULT:
{
  "count": 10,
  "total_count": 200,
  "offset": 0,
  "has_more": true,
  "next_offset": 10,
  "indicators": [
    {
      "idno": "WB_WDI_SP_DYN_LE00_IN",
      "database_id": "WB_WDI",
      "database_name": "World Development Indicators (WDI)",
      "name": "Life expectancy at birth, total (years)",
      "truncated_definition": "Life expectancy at birth indicates the number of years a newborn infant would live if prevailing pat",
      "periodicity": "Annual",
      "latest_data": "2024",
      "time_period_range": "1960-2024",
      "covers_country": {
        "EGY": true
      },
      "requested_country": "EGY",
      "dimensions": null,
      "primary_source_of": null
    },
    {
      "idno": "WB_WDI_SP_DYN_LE00_MA_IN",
      "database_id": "WB_WDI",
      "database_name": "World Development Indicators (WDI)",
      "name": "Life expectancy at birth, male (years)",
      "truncated_definiti

In [6]:

from agentic_framework.mcp_client import (
    MCP_URL,
    call_data360_tool,
    fetch_all_observations,
)
from mcp import Client

verified_cases = {
    "life_expectancy": {
        "database_id": "WB_WDI",
        "indicator_id": "WB_WDI_SP_DYN_LE00_IN",
        "country": "EGY",
        "start_year": 2000,
        "end_year": 2023,
    },
    "co2_emissions": {
        "database_id": "WB_WDI",
        "indicator_id": "WB_WDI_EN_GHG_CO2_PC_CE_AR5",
        "country": "FRA",
        "start_year": 2000,
        "end_year": 2023,
    },
    "unemployment": {
        "database_id": "WB_WDI",
        "indicator_id": "WB_WDI_SL_UEM_TOTL_ZS",
        "country": "CHE",
        "start_year": 2010,
        "end_year": 2023,
    },
}

verified_results = {}

async with Client(MCP_URL) as client:
    for name, case in verified_cases.items():
        print(f"\n{'=' * 60}")
        print("CASE:", name)

        metadata = await call_data360_tool(
            client,
            "data360_get_metadata",
            {
                "database_id": case["database_id"],
                "indicator_id": case["indicator_id"],
                "required_country": case["country"],
            },
        )

        record = metadata["indicator_metadata"]

        print("Indicator:", record["idno"])
        print("Name:", record["name"])
        print("Unit:", record["measurement_unit"])
        print("Periodicity:", record["periodicity"])

        observations = await fetch_all_observations(
            client,
            database_id=case["database_id"],
            indicator_id=case["indicator_id"],
            country_code=case["country"],
            start_year=case["start_year"],
            end_year=case["end_year"],
        )

        records = observations["data"]

        print("Observations:", len(records))
        print("Pages:", len(observations["raw_pages"]))
        print("First 3 records:", records[:3])

        verified_results[name] = {
            "metadata": metadata,
            "observations": observations,
        }



CASE: life_expectancy
Indicator: WB_WDI_SP_DYN_LE00_IN
Name: Life expectancy at birth, total (years)
Unit: Years
Periodicity: Annual
Observations: 24
Pages: 1
First 3 records: [{'OBS_VALUE': '71.633', 'UNIT_MULT': 0, 'REF_AREA': 'EGY', 'TIME_PERIOD': '2023', 'UNIT_MEASURE': 'YR', 'claim_id': '27451768', 'REF_AREA_NAME': 'Egypt, Arab Rep.', 'UNIT_MEASURE_NAME': 'Years'}, {'OBS_VALUE': '71.01', 'UNIT_MULT': 0, 'REF_AREA': 'EGY', 'TIME_PERIOD': '2022', 'UNIT_MEASURE': 'YR', 'claim_id': '050a3310', 'REF_AREA_NAME': 'Egypt, Arab Rep.', 'UNIT_MEASURE_NAME': 'Years'}, {'OBS_VALUE': '68.976', 'UNIT_MULT': 0, 'REF_AREA': 'EGY', 'TIME_PERIOD': '2021', 'UNIT_MEASURE': 'YR', 'claim_id': '6465627e', 'REF_AREA_NAME': 'Egypt, Arab Rep.', 'UNIT_MEASURE_NAME': 'Years'}]

CASE: co2_emissions
Indicator: WB_WDI_EN_GHG_CO2_PC_CE_AR5
Name: Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita)
Unit: t CO2e/capita
Periodicity: Annual
Observations: 24
Pages: 1
First 3 records: [{'OBS_VALU